In [0]:
%sql
-- CELL 1: Silver is designed to create a real table, written in SQL as a prototype.

CREATE OR REPLACE TABLE cms.silver.partd_prescriber_drug
COMMENT 'Part D prescriber-by-drug rows, typed and renamed. Suppressed values are NULL, never 0. Keep in mind, this is based on Medicare D Dictionary.'
AS
SELECT
  -- prescriber
  Prscrbr_NPI                              AS prescriber_npi,
  Prscrbr_Last_Org_Name                    AS prescriber_last_or_org_name,
  Prscrbr_First_Name                       AS prescriber_first_name,
  Prscrbr_First_Name IS NULL               AS is_organization,
  Prscrbr_City                             AS prescriber_city,
  Prscrbr_State_Abrvtn                     AS state_code,
  Prscrbr_State_FIPS                       AS state_fips,
  Prscrbr_Type                             AS specialty,
  Prscrbr_Type_Src                         AS specialty_source,

  -- drug
  Brnd_Name                                AS brand_name,
  Gnrc_Name                                AS generic_name,

  -- all-ages totals (claims and cost are never suppressed)
  CAST(Tot_Clms        AS BIGINT)          AS total_claims,
  CAST(Tot_30day_Fills AS DECIMAL(18,1))   AS total_30day_fills,
  CAST(Tot_Day_Suply   AS BIGINT)          AS total_day_supply,
  CAST(Tot_Drug_Cst    AS DECIMAL(18,2))   AS total_drug_cost,
  CAST(Tot_Benes       AS BIGINT)          AS total_beneficiaries,
  Tot_Benes IS NULL                        AS beneficiaries_suppressed,

  -- age 65+ totals
  CAST(GE65_Tot_Clms        AS BIGINT)        AS ge65_claims,
  CAST(GE65_Tot_30day_Fills AS DECIMAL(18,1)) AS ge65_30day_fills,
  CAST(GE65_Tot_Day_Suply   AS BIGINT)        AS ge65_day_supply,
  CAST(GE65_Tot_Drug_Cst    AS DECIMAL(18,2)) AS ge65_drug_cost,
  CASE GE65_Sprsn_Flag
    WHEN '*' THEN 'value_1_to_10'
    WHEN '#' THEN 'under65_value_1_to_10'
  END                                         AS ge65_suppression_reason,
  CAST(GE65_Tot_Benes AS BIGINT)              AS ge65_beneficiaries,
  CASE GE65_Bene_Sprsn_Flag
    WHEN '*' THEN 'value_1_to_10'
    WHEN '#' THEN 'under65_value_1_to_10'
  END                                         AS ge65_bene_suppression_reason,

  -- lineage
  data_year,
  _source_file,
  _ingested_at
FROM cms.bronze.partd_prescriber_drug;

-- Basically just renaming everything as what the dictionary says they are and categorizing the data. Plain CAST stops the entire statement on the first bad value, rather than making it a null. 

In [0]:
## CELL 2: Repeat of Bronze Cells 6 and 7.

silver_n = spark.table("cms.silver.partd_prescriber_drug").count()
bronze_n = spark.table("cms.bronze.partd_prescriber_drug").count()

c = spark.sql("""
SELECT
  SUM(CASE WHEN total_claims < 11 THEN 1 ELSE 0 END)                         AS below_11_claims,
  SUM(CASE WHEN total_30day_fills < total_claims
             OR total_30day_fills > 12 * total_claims THEN 1 ELSE 0 END)     AS fill_bounds_broken,
  SUM(CASE WHEN total_drug_cost IS NULL THEN 1 ELSE 0 END)                   AS null_cost,
  SUM(CASE WHEN (ge65_suppression_reason IS NOT NULL)
             <> (ge65_claims IS NULL) THEN 1 ELSE 0 END)                     AS ge65_flag_mismatch,
  SUM(CASE WHEN (ge65_suppression_reason IS NOT NULL)
             <> (ge65_drug_cost IS NULL) THEN 1 ELSE 0 END)                  AS ge65_cost_flag_mismatch,
  SUM(CASE WHEN (ge65_bene_suppression_reason IS NOT NULL)
             <> (ge65_beneficiaries IS NULL) THEN 1 ELSE 0 END)              AS ge65_bene_flag_mismatch
FROM cms.silver.partd_prescriber_drug
""").first()

assert silver_n == bronze_n,              f"Row count changed: bronze {bronze_n}, silver {silver_n}"
assert c.below_11_claims == 0,            f"{c.below_11_claims} rows under the 11-claim floor"
assert c.fill_bounds_broken == 0,         f"{c.fill_bounds_broken} rows outside claims <= fills <= 12x claims"
assert c.null_cost == 0,                  f"{c.null_cost} rows missing drug cost"
assert c.ge65_flag_mismatch == 0,         f"{c.ge65_flag_mismatch} rows where 65+ claims and its flag disagree"
assert c.ge65_cost_flag_mismatch == 0,    f"{c.ge65_cost_flag_mismatch} rows where 65+ cost and its flag disagree"
assert c.ge65_bene_flag_mismatch == 0,    f"{c.ge65_bene_flag_mismatch} rows where 65+ patients and its flag disagree"

print(f"Silver checks passed: {silver_n:,} rows")

## Literally just stops the notebook if any fails happen. 